In [43]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

In [6]:
dataset = load_dataset("PolyAI/banking77")

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names

train_df["intent"] = train_df["label"].apply(lambda x: label_names[x])
test_df["intent"] = test_df["label"].apply(lambda x: label_names[x])

Found cached dataset banking77 (C:/Users/alaaj/.cache/huggingface/datasets/PolyAI___banking77/default/1.1.0/17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9)


  0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
print(train_df.shape)
print(test_df.shape)

(10003, 3)
(3080, 3)


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

stratify preserves the probability of every lable in train_df , X_train and X_test

In [10]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(test_df))

Training samples: 8002
Validation samples: 2001
Test samples: 3080


In [12]:
original_distribution = train_df["label"].value_counts(normalize=True).sort_index()
original_distribution

0     0.015895
1     0.010997
2     0.012596
3     0.008697
4     0.012696
        ...   
72    0.004099
73    0.013496
74    0.012096
75    0.017995
76    0.016295
Name: label, Length: 77, dtype: float64

In [13]:
train_distribution = y_train.value_counts(normalize=True).sort_index()
train_distribution

0     0.015871
1     0.010997
2     0.012622
3     0.008748
4     0.012747
        ...   
72    0.004124
73    0.013497
74    0.012122
75    0.017996
76    0.016246
Name: label, Length: 77, dtype: float64

In [14]:
val_distribution = y_val.value_counts(normalize=True).sort_index()
val_distribution

0     0.015992
1     0.010995
2     0.012494
3     0.008496
4     0.012494
        ...   
72    0.003998
73    0.013493
74    0.011994
75    0.017991
76    0.016492
Name: label, Length: 77, dtype: float64

In [15]:
distribution_comparison = pd.DataFrame({
    "original": original_distribution,
    "train": train_distribution,
    "validation": val_distribution
})

distribution_comparison.head(10)

,original,train,validation
0,0.015895,0.015871,0.015992
1,0.010997,0.010997,0.010995
2,0.012596,0.012622,0.012494
3,0.008697,0.008748,0.008496
4,0.012696,0.012747,0.012494
5,0.017095,0.017121,0.016992
6,0.018095,0.018120,0.017991
7,0.015595,0.015621,0.015492
8,0.015695,0.015746,0.015492
9,0.012896,0.012872,0.012994


`stratify=train_df["label"]` preserves approximately the same
class proportions in the training and validation sets as in the
original dataset.

This is useful because BANKING77 contains 77 classes with different
numbers of examples. Without stratification, some smaller classes could
be underrepresented in one of the splits.

In [16]:
train_df["word_count"] = train_df["text"].str.split().str.len()

avg_words_by_intent = (
    train_df.groupby("intent")["word_count"]
    .mean()
    .sort_values(ascending=False)
)

In [19]:
avg_words_by_intent.head(10)


intent
transfer_not_received_by_recipient         17.549708
transfer_fee_charged                       17.395349
pending_cash_withdrawal                    16.531469
failed_transfer                            15.992701
direct_debit_payment_not_recognised        15.989011
transaction_charged_twice                  15.988571
Refund_not_showing_up                      15.808642
wrong_exchange_rate_for_cash_withdrawal    15.460123
cash_withdrawal_not_recognised             15.437500
wrong_amount_of_cash_received              15.283333
Name: word_count, dtype: float64

In [20]:
avg_words_by_intent.tail(10)

intent
edit_personal_details    8.471074
verify_top_up            8.309524
visa_or_mastercard       8.155556
order_physical_card      7.875000
exchange_rate            7.830357
top_up_limits            7.742268
get_physical_card        7.707547
passcode_forgotten       7.695238
atm_support              7.689655
card_acceptance          7.474576
Name: word_count, dtype: float64

Some intents tend to contain longer messages than others. For example,
`transfer_not_received_by_recipient` averages about 17.5 words, while
`card_acceptance` averages about 7.5 words.

This suggests that message length may contain some information about
the intent, although it is unlikely to be sufficient by itself for
classification.


In [21]:
all_words = " ".join(train_df["text"].str.lower()).split()

word_counts = Counter(all_words)

word_counts.most_common(30)

[('i', 8309),
 ('my', 5684),
 ('to', 4011),
 ('a', 3565),
 ('the', 3497),
 ('is', 2339),
 ('can', 1839),
 ('do', 1730),
 ('card', 1636),
 ('it', 1557),
 ('for', 1534),
 ('how', 1511),
 ('what', 1367),
 ('why', 1233),
 ('and', 1213),
 ('you', 1178),
 ('was', 1089),
 ('have', 975),
 ('not', 953),
 ('in', 944),
 ('money', 920),
 ('that', 883),
 ('transfer', 861),
 ('me', 845),
 ('there', 827),
 ('on', 819),
 ('get', 796),
 ('up', 765),
 ('an', 722),
 ('this', 714)]

The most frequent words are not necessarily the most useful words for
classification.

Words such as "I", "my", "to", and "the" appear in many different
intent classes and therefore provide little information for
distinguishing between them.

A word can be highly frequent but have low discriminative power.

In contrast, terms such as "PIN", "withdrawal", "refund", or "transfer"
may occur less often overall but can be much more informative for
predicting a particular intent.

In [23]:
selected = train_df[
    train_df["intent"] == "change_pin"
]["text"]

In [24]:
selected

6883       Is it possible for me to change my PIN number?
6884    What are the steps to change my PIN to somethi...
6885    In what way can I change my PIN and where do I...
6886            Can I change my PIN at any cash machines?
6887        I need to make my card PIN a different number
                              ...                        
7000                 Please tell me how to change my pin.
7001                     At what ATM can I change my PIN?
7002    If I am not in the country and I need to chang...
7003               What do I have to do to change my pin?
7004        Can I change my pin number at a cash machine?
Name: text, Length: 122, dtype: object

In [50]:
selected2 = train_df[
    train_df["intent"] == "cash_withdrawal_not_recognised"
]["text"]
selected2

8784    I looked on the app and it says I withdrew cas...
8785    I didn't withdraw the amount of cash that is s...
8786    I saw on the app that a cash withdrawal was co...
8787    I got a notice from my app that I withdrew cas...
8788    Someone has a copy of my card !! I definitely ...
                              ...                        
8939    A cash withdrawal that I didn't authorize is s...
8940           Cash I did not get showed up in my account
8941    My app says I withdraw funds from my account t...
8942    There has been an unusual withdrawal from my a...
8943               Some cash I didn't get shows in my app
Name: text, Length: 160, dtype: object

In [51]:
selected3 = train_df[
    train_df["intent"] == "card_arrival"
]["text"]
selected3

0                         I am still waiting on my card?
1      What can I do if my card still hasn't arrived ...
2      I have been waiting over a week. Is the card s...
3      Can I track my card while it is in the process...
4      How do I know if I will get my card, or if it ...
                             ...                        
148         I haven't gotten my credit card in the mail.
149            My card hasn't arrived yet. What do I do?
150    My card still hasn't arrived after two weeks, ...
151               My card appears to have never arrived?
152                               My card never arrived.
Name: text, Length: 153, dtype: object

there are words that are repeated multble time for eatch intent , for example for "change_pin" you find 'change my pin' ,for the "card_arrival" you find 'arrived'

In [41]:
dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(
    np.zeros((len(y_train), 1)),
    y_train
)

DummyClassifier(strategy='most_frequent')

The `most_frequent` dummy classifier always predicts the class that
appears most frequently in the training data, regardless of the input
text.

In [42]:
dummy_predictions = dummy.predict(
    np.zeros((len(y_val), 1))
)

In [44]:
dummy_accuracy = accuracy_score(
    y_val,
    dummy_predictions
)

dummy_macro_f1 = f1_score(
    y_val,
    dummy_predictions,
    average="macro"
)

print("Dummy Accuracy:", dummy_accuracy)
print("Dummy Macro F1:", dummy_macro_f1)

Dummy Accuracy: 0.01899050474762619
Dummy Macro F1: 0.00048406718342961603


| Model                        |    Accuracy |    Macro F1 |
| ---------------------------- | ----------: | ----------: |
| Dummy Most Frequent          | 0.018990504 | 0.000484067 |
| TF-IDF + Logistic Regression |       later |       later |
| TF-IDF + SVM                 |       later |       later |
| Transformer                  |  much later |  much later |


The accuracy is low because BANKING77 does not contain one overwhelmingly
dominant class. Predicting only the largest class therefore gives the
correct prediction for only about 1.9% of validation examples.

The Macro-F1 is extremely low because the classifier predicts only one
class.

The majority class receives a non-zero F1 score, while the remaining
76 classes receive an F1 score of zero.

Since Macro-F1 gives equal importance to every class, averaging these
77 class-level F1 scores results in a value close to zero.

In [45]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_val,
    dummy_predictions
)

cm.shape

(77, 77)

In [46]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val,
        dummy_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        32
           1       0.00      0.00      0.00        22
           2       0.00      0.00      0.00        25
           3       0.00      0.00      0.00        17
           4       0.00      0.00      0.00        25
           5       0.00      0.00      0.00        34
           6       0.00      0.00      0.00        36
           7       0.00      0.00      0.00        31
           8       0.00      0.00      0.00        31
           9       0.00      0.00      0.00        26
          10       0.00      0.00      0.00        12
          11       0.00      0.00      0.00        31
          12       0.00      0.00      0.00        22
          13       0.00      0.00      0.00        28
          14       0.00      0.00      0.00        22
          15       0.02      1.00      0.04        38
          16       0.00      0.00      0.00        34
          17       0.00    

lable 15 has 0.02 precision , 1 recall , and 0.04 f1 score , and all the others has 0 , because the lable 15 is the biggest

In [47]:
X_test = test_df["text"]
y_test = test_df["label"]

In [49]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(8002,)
(2001,)
(3080,)


## Experimental Strategy

The original BANKING77 training set is divided into training and validation subsets using stratified sampling.

The training set is used to fit models.

The validation set is used for model comparison, feature choices, and hyperparameter selection.

The original test set is kept separate and will only be used for final evaluation after the model-development decisions have been made.

This avoids using the test set repeatedly during development and provides a more reliable estimate of model generalization.

### Training set

Used to fit the model and learn its parameters.

### Validation set

Used during model development to compare models, tune
hyperparameters, and make feature/model-selection decisions.

### Test set

Kept separate during development and used at the end to estimate how
well the final selected model generalizes to unseen data.


### stratified split

we are using a stratified split to have same distrubition of lables in the training and validation set

### repeated test evaluation

If we repeatedly evaluate different models on the test set and choose
models based on those results, information from the test set indirectly
influences our decisions.

Over time, we may optimize our model-selection process for that
particular test set.

The resulting test score would then be an overly optimistic estimate of
performance on genuinely unseen data.

### baseline model?

to compare it to other models and to see the improvement 


### accuracy misleading

Why can accuracy be misleading for an imbalanced classification dataset?

Accuracy can be misleading when one class is much more frequent than
the others.

For example, if 99% of samples belong to class A, a classifier that
always predicts class A obtains 99% accuracy even though it completely
fails to identify class B.

Therefore, accuracy should be considered together with metrics such as
precision, recall, F1-score, Macro-F1, and per-class performance.

### Macro-F1

Macro-F1 is useful for this 77-class problem because it calculates the
F1-score for each class independently and then gives every class equal
weight.

This prevents performance on larger classes from hiding poor
performance on smaller classes.

It therefore helps us evaluate whether the classifier performs
reasonably well across all 77 intents.


### hypothesis 

Yes, I expect TF-IDF with a linear classifier to perform reasonably
well.

The BANKING77 messages are short, averaging roughly 12 words, and many
intents contain discriminative vocabulary such as "PIN", "refund",
"withdrawal", "card", or "transfer".

TF-IDF can assign higher importance to words and phrases that are
specific to certain intents.

However, TF-IDF may struggle when different intents use similar
vocabulary or when users express the same meaning using different
words.



